<a href="https://colab.research.google.com/github/Elwing-Chou/tibame20260427/blob/main/tibame20260606.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


```
上次的複習(爬蟲整個流程)

1. 找到網址: a. 原始碼有  b. 原始碼沒有(F12找隱藏)
2. 解析看格式: a. JSON格式  b. HTML格式

分支1. JSON格式
1. json.loads: 字典/list解析

分支2. HTML格式
2. BeautifulSoup: 找你需要的區塊(html名字+class做篩選)

排版屬性: 在區塊上加上這些屬性來輔助排版
a. class: class="分類1 分類2"(熟悉)
b. id: id="xxx"(補充)
html.find(名字, {"id":"xxxx"})

3. 儲存+分析(pandas)


```


In [28]:
import urllib.request as req
import bs4 as bs

def get_page(page_num):
    url = f"https://tabelog.com/tw/tokyo/rstLst/sweets/{page_num}/?SrtT=rt"
    print(url)
    resp = req.urlopen(url)
    content = resp.read()
    html = bs.BeautifulSoup(content)
    rs = html.find_all("div", {"class":"list-rst__body"})

    # 最後回傳的一頁的餐廳清單
    result = []
    for r in rs:
        name = r.find("a", {"class":"list-rst__rst-name-target"})
        area_genre = r.find("div", {"class":"list-rst__area-genre"})
        rating = r.find("span", {"class":"c-rating__val"})
        prices = r.find_all("span", {"class":"c-rating-v3__val"})
        dinner_price = prices[0]
        lunch_price = prices[1]
        holiday = r.find("span", {"class":"list-rst__holiday-text"})
        imgs = r.find_all("img", {"class":"js-thumbnail-img"})

        name_text = name.get_text().strip()
        name_href = name["href"]
        area_genre_text = area_genre.get_text().strip()
        rating_text = rating.get_text().strip()
        dinner_price_text = dinner_price.get_text().strip()
        lunch_price_text = lunch_price.get_text().strip()
        holiday_text = holiday.get_text().strip()

        imgs_src = []
        for img in imgs:
            imgs_src.append(img["data-lazy"])

        # 一筆資料是一個字點
        data = {
            "rating":rating_text,
            "name":name_text,
            "link":name_href,
            "area_genre":area_genre_text,
            "price_dinner":dinner_price_text,
            "price_lunch":lunch_price_text,
            "holiday":holiday_text,
            "imgs":imgs_src,
        }
        result.append(data)

    return result

get_page(2)

https://tabelog.com/tw/tokyo/rstLst/sweets/2/?SrtT=rt


[{'rating': '3.85',
  'name': 'patisserie K.ViNCENT',
  'link': 'https://tabelog.com/tw/tokyo/A1309/A130905/13035194/',
  'area_genre': '飯田橋車站 408m / 蛋糕, 巧克力, 西式甜點',
  'price_dinner': 'JPY 10,000 - JPY 14,999',
  'price_lunch': '-',
  'holiday': '-',
  'imgs': ['https://tblg.k-img.com/restaurant/images/Rvw/179334/320x320_square_5f70f42e1d93c672b31be6ae32a35eaa.jpg',
   'https://tblg.k-img.com/restaurant/images/Rvw/179334/320x320_square_4e4d44b4f111d6a1fb7b4dc942be4bd1.jpg',
   'https://tblg.k-img.com/restaurant/images/Rvw/4291/320x320_square_4291051.jpg']},
 {'rating': '3.85',
  'name': "L'AUTOMNE 中野店",
  'link': 'https://tabelog.com/tw/tokyo/A1321/A132101/13118933/',
  'area_genre': '新江古田車站 39m / 蛋糕, 咖啡店, 麵包',
  'price_dinner': 'JPY 1,000 - JPY 1,999',
  'price_lunch': 'JPY 1,000 - JPY 1,999',
  'holiday': '星期一, 星期二',
  'imgs': ['https://tblg.k-img.com/restaurant/images/Rvw/151045/320x320_square_151045029.jpg',
   'https://tblg.k-img.com/restaurant/images/Rvw/151045/320x320_square_151

In [53]:
# [{}餐廳, {}餐廳] -> dataframe
import pandas as pd

total = []
for i in range(5):
    page = i + 1
    partial = get_page(page)
    total = total + partial

df = pd.DataFrame(total)
df

https://tabelog.com/tw/tokyo/rstLst/sweets/1/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/2/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/3/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/4/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/5/?SrtT=rt


,rating,name,link,area_genre,price_dinner,price_lunch,holiday,imgs
0,4.24,Bon.nu,https://tabelog.com/tw/tokyo/A1304/A130401/131...,"參宮橋車站 356m / 法式料理, 牛排, 蛋糕","JPY 50,000 - JPY 59,999","JPY 50,000 - JPY 59,999",-,[https://tblg.k-img.com/restaurant/images/Rvw/...
1,4.20,Okashiya Ucchi,https://tabelog.com/tw/tokyo/A1309/A130901/132...,"北參道車站 250m / 蛋糕, 西式甜點",-,"JPY 5,000 - JPY 5,999",星期一,[https://tblg.k-img.com/restaurant/images/Rvw/...
2,4.16,蒼菓,https://tabelog.com/tw/tokyo/A1307/A130703/132...,廣尾車站 702m / 甜食,"JPY 15,000 - JPY 19,999","JPY 15,000 - JPY 19,999","星期一, 星期二, 星期日",[https://tblg.k-img.com/restaurant/images/Rvw/...
3,4.15,Yama,https://tabelog.com/tw/tokyo/A1316/A131602/132...,白金台車站 725m / 甜食,"JPY 20,000 - JPY 29,999","JPY 30,000 - JPY 39,999","星期二, 星期三",[https://tblg.k-img.com/restaurant/images/Rvw/...
4,4.05,ESPRIT C. KEI GINZA,https://tabelog.com/tw/tokyo/A1301/A130101/132...,"銀座車站 330m / 法式料理, 甜食","JPY 30,000 - JPY 39,999",-,"星期一, 星期日",[https://tblg.k-img.com/restaurant/images/Rvw/...
...,...,...,...,...,...,...,...,...
95,3.77,秘密堂,https://tabelog.com/tw/tokyo/A1311/A131106/131...,"日暮里車站 376m / 刨冰, 日式甜點店","JPY 2,000 - JPY 2,999","JPY 1,000 - JPY 1,999","星期一, 星期二",[https://tblg.k-img.com/restaurant/images/Rvw/...
96,3.77,Fruit Parlour Goto,https://tabelog.com/tw/tokyo/A1311/A131102/130...,"淺草車站 224m / 水果聖代, 咖啡店, 刨冰","JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999","星期三, 星期四",[https://tblg.k-img.com/restaurant/images/Rvw/...
97,3.77,MAISON LANDEMAINE 麻布台,https://tabelog.com/tw/tokyo/A1307/A130701/131...,"六本木一丁目車站 429m / 麵包, 西式甜點, 咖啡店","JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999",-,[https://tblg.k-img.com/restaurant/images/Rvw/...
98,3.77,FRENCH POUND HOUSE 大和鄉本店,https://tabelog.com/tw/tokyo/A1323/A132301/130...,"巢鴨車站 228m / 蛋糕, 咖啡店","JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999",-,[https://tblg.k-img.com/restaurant/images/Rvw/...


In [19]:
# demo: 如何下載一筆圖片
import urllib.request as req

url = "https://japan.videoland.com.tw/images/contents/20250821144057983.png"
h = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0"
}
r = req.Request(url, headers=h)

resp = req.urlopen(r)
content = resp.read()

# 開一個空白檔案, 把content寫進去
# 純文字: f = open(檔名, "w", encoding="utf-8") "r" or "w"
# 其它(jpg, jpeg, png, pdf, docx): f = open("檔名", "rb") "rb" or "wb"
f = open("a.png", "wb")
f.write(content)
f.close()

In [ ]:
# DataFrame: pandas表格型態(二維)
# Series: pandas list型態(一維)
import os
import time
import urllib.request as req

dn = "tabelog2"
if not os.path.exists(dn):
    os.makedirs(dn)
for d in df["imgs"]:
    for img in d:
        print(img)
        # 每一個圖片網址
        fp = f"{dn}/{time.time()}.png"
        h = {
            "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0"
        }
        r = req.Request(img, headers=h)
        resp = req.urlopen(r)
        content = resp.read()
        # 開一個新檔案, 把你從網址讀出來的內容寫入
        f = open(fp, "wb")
        f.write(content)
        f.close()


```
pandas 第二個你要會的

apply: 轉換

Python: Everything is an object
這句話只跟你說了一件事: 任何東西都有形態, 每個型態都有他的操作

List型態: [] 操作: 拿key/append/for...in/*
!!! 函式也有形態

函式: 把很多句程式組合在一起變成一個步驟
以前: randint(0, 2)
現在: 其實是分開的

步驟型態: randint  操作: (0, 2)
刷牙步驟        操作: ()

```



In [38]:
def add(n1, n2):
    n = n1 + n2
    n = round(n)
    return n

b = add
b(3.2, 5.6)

9

In [ ]:
# demo: Series.apply(錦囊)

demo = pd.DataFrame([
    [1, 2],
    [3, 4],
    [5, 6]
], columns=["a", "b"])
# 1. 使用已經有的錦囊
demo["a"].apply(float)
# 2. 自己定義功能
def func(n):
    import random
    n = n + random.uniform(-1, 1)
    return n
demo["a"].apply(func)

In [59]:
def func(s):
    # 如果你想把他變成多格, 回傳Series
    d = {
        "星期一":"y",
        "星期二":"y",
        "星期三":"y",
        "星期四":"y",
        "星期五":"y",
        "星期六":"y",
        "星期日":"y",
    }
    for k in d:
        if k in s:
            d[k] = "n"
    return pd.Series(d)

new = df["holiday"].apply(func)
# df[['星期一', '星期二', '星期三', '星期四', '星期五', '星期六', '星期日']] = new
# df
# 結合過濾
fil1 = new["星期六"] == "n"
fil2 = new["星期日"] == "n"
fil = fil1 & fil2
df[fil]

,rating,name,link,area_genre,price_dinner,price_lunch,holiday,imgs
83,3.78,清壽軒,https://tabelog.com/tw/tokyo/A1302/A130204/132...,"小傳馬町車站 319m / 日式點心, 銅鑼燒",-,- JPY 999,"星期六, 星期日, 節假日",[https://tblg.k-img.com/restaurant/images/Rvw/...
91,3.78,STYLE'S CAKES & CO.,https://tabelog.com/tw/tokyo/A1310/A131003/131...,神保町車站 232m / 蛋糕,"JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999","星期三, 星期六, 星期日, 節假日",[https://tblg.k-img.com/restaurant/images/Rvw/...


In [48]:
d = {
    "星期一":"y",
    "星期二":"y",
    "星期三":"y",
    "星期四":"y",
    "星期五":"y",
    "星期六":"y",
    "星期日":"y",
}
for k in d:
    print(k)

星期一
星期二
星期三
星期四
星期五
星期六
星期日
